# 01 — Лаборатория греков: кривые чувствительностей и греки позиции

Вы построите каждый грек как функцию спота и времени, а затем сложите греки по двухногой позиции.
Всё на DEMO: спот **$100**, IV **0.25**, в основном **45 DTE**.

Цели:
1. `greeks.bsm_greeks` — дельта/гамма/тета/вега против спота и против DTE.
2. `greeks.position_greeks` — греки реальных позиций, агрегированные в долларах.
3. `viz.plot_greeks` — кривые греков позиции.
4. Увидеть *дельта-нейтральность и шорт по веге* в числах короткого стрэддла.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import greeks, strategies, viz

SPOT, VOL = 100.0, 0.25
t = 45 / 365

## 1. Греки одного ATM-колла

`bsm_greeks` возвращает датакласс `Greeks` (delta, gamma, theta, vega, rho) в расчёте на акцию.

In [ ]:
g = greeks.bsm_greeks('call', SPOT, strike=100, t=t, vol=VOL)
print(f'дельта {g.delta:.3f}  гамма {g.gamma:.4f}  тета {g.theta:.4f}  вега {g.vega:.3f}  ро {g.rho:.3f}')

## 2. Дельта против спота — от 0 до 1 у коллов, от 0 до −1 у путов

Пройдём по споту при страйке, зафиксированном на 100. Смотрите, как дельта колла проходит 0.5 на
деньгах и стремится к 1 в деньгах.

In [ ]:
spots = np.linspace(70, 130, 121)
call_delta = [greeks.bsm_greeks('call', s, 100, t, VOL).delta for s in spots]
put_delta  = [greeks.bsm_greeks('put',  s, 100, t, VOL).delta for s in spots]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(spots, call_delta, label='дельта колла')
ax.plot(spots, put_delta, label='дельта пута')
ax.axvline(100, color='k', ls='--', lw=1); ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('спот'); ax.set_ylabel('дельта'); ax.set_title('Дельта против спота (страйк 100, 45 DTE)'); ax.legend()
plt.show()

## 3. Гамма, тета и вега — все пикуют на деньгах

Построим против спота три грека, максимальных на ATM. Заметьте: гамма и вега положительны (длинный
опцион), а тета отрицательна (вы платите за распад).

In [ ]:
gam = [greeks.bsm_greeks('call', s, 100, t, VOL).gamma for s in spots]
the = [greeks.bsm_greeks('call', s, 100, t, VOL).theta for s in spots]
veg = [greeks.bsm_greeks('call', s, 100, t, VOL).vega  for s in spots]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, y, name in zip(axes, [gam, the, veg], ['гамма', 'тета', 'вега']):
    ax.plot(spots, y); ax.axvline(100, color='k', ls='--', lw=1); ax.set_title(name); ax.set_xlabel('спот')
plt.tight_layout(); plt.show()

## 4. Гамма взрывается, а вега угасает по мере приближения экспирации

Зафиксируем ATM-страйк и будем сокращать DTE. Гамма резко растёт у экспирации; вега уменьшается.
Это дилемма продавца на одном графике.

In [ ]:
dtes = np.array([90, 60, 45, 30, 21, 14, 7, 3, 1])
atm_gamma = [greeks.bsm_greeks('call', 100, 100, d/365, VOL).gamma for d in dtes]
atm_vega  = [greeks.bsm_greeks('call', 100, 100, d/365, VOL).vega  for d in dtes]
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(dtes, atm_gamma, 'o-', color='C1', label='гамма'); ax1.set_xlabel('DTE'); ax1.set_ylabel('гамма', color='C1')
ax2 = ax1.twinx(); ax2.plot(dtes, atm_vega, 's-', color='C0', label='вега'); ax2.set_ylabel('вега', color='C0')
ax1.invert_xaxis(); ax1.set_title('ATM: гамма растёт, вега падает к экспирации'); plt.show()

## 5. Греки позиции: бычий колл-спред

Соберём бычий колл-спред DEMO 100/110 и агрегируем его греки **в долларах** (с применённым
`quantity × multiplier`). Сравним с голым длинным коллом 100.

In [ ]:
spread = strategies.bull_call_spread((100, 3.91), (110, 0.73), expiry=t)
lone   = strategies.long_call((100, 3.91), expiry=t)
gs = greeks.position_greeks(spread, SPOT, VOL)
gl = greeks.position_greeks(lone,   SPOT, VOL)
print('спред       ', f'дельта {gs.delta:7.1f}  тета {gs.theta:7.2f}  вега {gs.vega:7.2f}')
print('длинный колл', f'дельта {gl.delta:7.1f}  тета {gl.theta:7.2f}  вега {gl.vega:7.2f}')

У спреда **дельта меньше**, чем у одиночного колла (короткая нога 110 вычитает направление), и при
этом **меньше веги и меньше теты** — продажа колла 110 подрезает те греки, за которые вы не хотели
платить.

## 6. Дельта-нейтральный, шорт по веге, лонг по тете: короткий стрэддл

Продадим колл 100 и пут 100. Дельты почти гасят друг друга (дельта-нейтральность); обе ноги
короткие, поэтому вега отрицательна (шорт по веге), а тета положительна (вы собираете распад).

In [ ]:
straddle = strategies.short_straddle((100, 3.91), (100, 3.42), expiry=t)
gstr = greeks.position_greeks(straddle, SPOT, VOL)
print(f'дельта {gstr.delta:7.2f}  (около нуля => дельта-нейтрально)')
print(f'гамма  {gstr.gamma:7.3f}  (отрицательна => шорт по гамме, опасно на больших движениях)')
print(f'тета   {gstr.theta:7.2f}  (положительна => собираем распад каждый день)')
print(f'вега   {gstr.vega:7.2f}  (отрицательна => шорт по веге, прибыль при падении IV)')

## 7. Атрибуция P&L: греки против точной переоценки

Разложим ночной P&L длинного колла 100 через оценку Тейлора
`delta*dS + 0.5*gamma*dS^2 + theta*dt + vega*(dIV в пунктах)`, а затем сравним с точной переоценкой
через `bsm_price`. На малых движениях они близко совпадают; гамма — та кривизна, которая закрывает
разрыв.

In [ ]:
from optionslab import pricing
g0 = greeks.bsm_greeks('call', 100, 100, t, VOL)
dS, dIV, ddays = 2.0, -0.01, 1          # спот +$2, IV −1 пункт, проходит 1 день
est = g0.delta*dS + 0.5*g0.gamma*dS**2 + g0.theta*ddays + g0.vega*(dIV/0.01)
p0 = pricing.bsm_price('call', 100, 100, t, VOL)
p1 = pricing.bsm_price('call', 100+dS, 100, t - ddays/365, VOL + dIV)
print(f'оценка по грекам   {est:+.3f} / акцию')
print(f'точная переоценка  {p1 - p0:+.3f} / акцию')

## 8. Кривые греков позиции против спота

`viz.plot_greeks` показывает, как долларовые греки позиции меняются при движении спота. У короткого
стрэддла смотрите, как дельта пересекает ноль на 100, а гамма всюду остаётся отрицательной.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_greeks(straddle, vol=VOL, which=('delta', 'gamma'), ax=ax)
ax.set_title('Короткий стрэддл: дельта и гамма позиции против спота')
plt.show()

## Эксперименты

1. В разделе 4 повторите проход по DTE для **теты** вместо веги. Убедитесь, что модуль теты растёт
   к экспирации — зеркало распада временной стоимости из модуля 00.
2. В разделе 5 расширьте спред до 100/120 (продайте колл 120 примерно за 0.07). Что произойдёт с
   чистой дельтой и чистой вегой по сравнению с версией 100/110?
3. В разделе 6 превратите стрэддл в **стрэнгл**: продайте пут 95 и колл 105. Он всё ещё
   дельта-нейтрален? Как гамма и тета соотносятся со стрэддлом?
4. Перезапустите раздел 2 на 7 DTE вместо 45. Насколько *резче* стал переход дельты через страйк
   (эффект гаммы)?
5. Посчитайте `position_greeks` для одиночного **короткого** пута 100. Убедитесь, что его дельта
   положительна, а вега отрицательна — зеркало длинного пута.